# Gemini 3 Flash Preview — Data Collection (Repeatability + Reproducibility)

This notebook mirrors the **exact collection protocol** I used for GPT-5, so I can describe one single methodology in the paper. The only differences from the GPT-5 notebook are (a) the API client (Google Gen AI SDK), (b) the model ID, and (c) the output file paths.

**Protocol (identical to GPT-5 collection):**
- Model: `gemini-3-flash-preview` (Google's current fast-tier Gemini model; preview release — verify at ai.google.dev)
- Temperature: 1.0
- Runs per prompt / variant: 30
- Max concurrent requests: 5 (asyncio semaphore)
- 60-second sleep between runs within the same prompt/variant
- Up to 6 retry attempts per API call, 10-second sleep between retries
- Continuation logic if the response is cut off (up to 2 extra calls with "Continue.")
- Parallel across prompts/variants; sequential across runs within a prompt/variant


**Before running:**
1. `pip install google-genai`
2. Add `GOOGLE_API_KEY=AIza...` to the `.env` file (get one at https://aistudio.google.com/apikey)
3. Add billing to your Google Cloud account for sufficient rate limits (Flash tier recommended for throughput)

> **Reviewer note:** To reproduce this notebook, update `PROJECT_DIR` to your local path. API keys are read from a `.env` file in `PROJECT_DIR` (see the `.env.example` in the repository root).

## 1. Imports and prompts

In [18]:
import sys, aiohttp
print(sys.executable)
print(aiohttp.__version__)

/Users/luizzadallapria/.venv/bin/python
3.14.4


In [19]:
!pip install google-genai
from google import genai
from google.genai import types
import pandas as pd
import time
import os
import csv
import datetime
import asyncio
from pathlib import Path

def load_env(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if '=' in line:
                k, v = line.split('=', 1)
                k = k.strip()
                if k.lower() == 'openai_api_key':
                    k = 'OPENAI_API_KEY'
                os.environ[k] = v.strip().strip('"').strip("'")

PROJECT_DIR = Path('/Users/luizzadallapria/Documents/GPT5_Study')
load_env(PROJECT_DIR / '.env')
assert os.environ.get('GOOGLE_API_KEY'), 'GOOGLE_API_KEY missing from .env'
print('API key loaded ✓')

API key loaded ✓


In [20]:
df_prompts = pd.read_csv(PROJECT_DIR / 'Prompts_to_GPT' / 'repeatability_prompts copy.csv')
prompts = list(df_prompts.itertuples(index=False, name=None))
print(f'Loaded {len(prompts)} repeatability prompts')

df_variants = pd.read_csv(PROJECT_DIR / 'Prompts_to_GPT' / 'reproductibility_prompts copy.csv')
PROMPTS_VARIANTS = (
    df_variants
    .groupby('prompt_id')
    .apply(lambda g: list(zip(g['variant_id'], g['variant_text'])))
    .to_dict()
)
print(f'Loaded {sum(len(v) for v in PROMPTS_VARIANTS.values())} reproducibility variants across {len(PROMPTS_VARIANTS)} prompts')

Loaded 15 repeatability prompts
Loaded 75 reproducibility variants across 15 prompts


## 2. Config, client, helpers

In [22]:
# ---------- CONFIG ----------
MODEL_NAME              = 'gemini-3-flash-preview'
TEMPERATURE             = 1.0
NUM_RUNS                = 30
MAX_CONCURRENT_REQUESTS = 5
MAX_OUTPUT_TOKENS       = 4096
MAX_CONTINUES           = 2
REQUEST_TIMEOUT         = 200
MAX_RETRIES             = 6
RETRY_SLEEP_SECONDS     = 10
INTER_RUN_SLEEP_SECONDS = 60

OUTPUT_DIR = PROJECT_DIR / 'Outputs_From_Gemini'
OUTPUT_DIR.mkdir(exist_ok=True)
REPEAT_CSV = OUTPUT_DIR / 'repeatability_outputs_gemini.csv'
REPRO_CSV  = OUTPUT_DIR / 'reproducibility_outputs_gemini.csv'

client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])

REPEAT_HEADER = [
    'prompt_id','prompt_text','run_id','model','temperature',
    'timestamp_utc','latency_ms','prompt_tokens','completion_tokens',
    'total_tokens','finish_reason','answer_content',
]
REPRO_HEADER = [
    'prompt_id','variant_id','variant_text','run_id','model','temperature',
    'timestamp_utc','latency_ms','prompt_tokens','completion_tokens',
    'total_tokens','finish_reason','answer_content',
]
for path, header in [(REPEAT_CSV, REPEAT_HEADER), (REPRO_CSV, REPRO_HEADER)]:
    if not path.exists():
        with open(path, 'w', newline='', encoding='utf-8') as f:
            csv.writer(f, quoting=csv.QUOTE_ALL).writerow(header)
print('Config and CSVs ready ✓')

Config and CSVs ready ✓


In [23]:
# ---------- extract text from Gemini response ----------
def _extract_text(resp):
    """Return the plain-text content from a Gemini response."""
    try:
        return (resp.text or '').strip()
    except Exception:
        parts = []
        if resp.candidates:
            for cand in resp.candidates:
                if cand.content and cand.content.parts:
                    for p in cand.content.parts:
                        if hasattr(p, 'text') and isinstance(p.text, str):
                            parts.append(p.text)
        return '\n'.join(parts).strip()

# ---------- one logical call, with continuation if cut off ----------
async def call_model(prompt_text: str, sem: asyncio.Semaphore):
    """
    Mirrors GPT-5 call_model: one call, with continuation if the response ends
    with 'MAX_TOKENS'. Uses Gemini's multi-turn chat format for continuation.
    """
    t0 = time.time()
    history = [types.Content(role='user', parts=[types.Part.from_text(text=prompt_text)])]
    collected = []
    total_prompt_tokens = 0
    total_completion_tokens = 0
    total_tokens = 0
    finish_reason_name = None

    async with sem:
        for _ in range(MAX_CONTINUES + 1):
            resp = await client.aio.models.generate_content(
                model=MODEL_NAME,
                contents=history,
                config=types.GenerateContentConfig(
                    temperature=TEMPERATURE,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                ),
            )
            text = _extract_text(resp)
            collected.append(text)

            usage = resp.usage_metadata
            total_prompt_tokens     += getattr(usage, 'prompt_token_count', 0) or 0
            total_completion_tokens += getattr(usage, 'candidates_token_count', 0) or 0
            total_tokens             = getattr(usage, 'total_token_count',
                                               total_prompt_tokens + total_completion_tokens) or (
                                       total_prompt_tokens + total_completion_tokens)

            if resp.candidates:
                fr = resp.candidates[0].finish_reason
                finish_reason_name = fr.name if hasattr(fr, 'name') else str(fr)
            else:
                finish_reason_name = 'UNKNOWN'

            if finish_reason_name != 'MAX_TOKENS':
                break

            # Continue the conversation (matches GPT-5 'Continue.' pattern)
            history.append(types.Content(role='model', parts=[types.Part.from_text(text=text)]))
            history.append(types.Content(role='user',  parts=[types.Part.from_text(text='Continue.')]))

    t1 = time.time()
    final_text = '\n'.join([c for c in collected if c]).strip()

    return {
        'answer': final_text,
        'finish_reason': finish_reason_name,
        'latency_ms': int((t1 - t0) * 1000),
        'prompt_tokens': total_prompt_tokens,
        'completion_tokens': total_completion_tokens,
        'total_tokens': total_tokens,
    }

print('Helpers defined ✓')

Helpers defined ✓


## 3. Repeatability — one task per prompt, 30 sequential runs with 60s spacing

In [24]:
async def run_prompt_repeatability(prompt_id, prompt_text, sem, csv_lock):
    print(f'\n  prompt {prompt_id}: collecting {NUM_RUNS} runs')
    for run_id in range(1, NUM_RUNS + 1):
        retries = 0
        while True:
            try:
                out = await call_model(prompt_text, sem)
                row = [
                    prompt_id, prompt_text, run_id,
                    MODEL_NAME, TEMPERATURE,
                    datetime.datetime.utcnow().isoformat(),
                    out['latency_ms'],
                    out['prompt_tokens'], out['completion_tokens'], out['total_tokens'],
                    out['finish_reason'], out['answer'],
                ]
                async with csv_lock:
                    with open(REPEAT_CSV, 'a', newline='', encoding='utf-8') as f:
                        csv.writer(f, quoting=csv.QUOTE_ALL).writerow(row)
                print(f'Saved prompt {prompt_id} run {run_id:02d} (tokens {out["total_tokens"]})')
                break
            except Exception as e:
                retries += 1
                print(f'Error prompt {prompt_id} run {run_id:02d} attempt {retries}: {e}')
                if retries >= MAX_RETRIES:
                    print(f'Giving up on prompt {prompt_id} run {run_id:02d} after {MAX_RETRIES} attempts')
                    break
                await asyncio.sleep(RETRY_SLEEP_SECONDS)
        await asyncio.sleep(INTER_RUN_SLEEP_SECONDS)

async def run_entire_repeatability():
    sem = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    csv_lock = asyncio.Lock()
    tasks = [asyncio.create_task(run_prompt_repeatability(int(pid), ptext, sem, csv_lock))
             for (pid, ptext) in prompts]
    await asyncio.gather(*tasks)
    print(f'\n✅ Repeatability done. Results in {REPEAT_CSV}')

await run_entire_repeatability()


  prompt 1: collecting 30 runs

  prompt 2: collecting 30 runs

  prompt 3: collecting 30 runs

  prompt 4: collecting 30 runs

  prompt 5: collecting 30 runs

  prompt 6: collecting 30 runs

  prompt 7: collecting 30 runs

  prompt 8: collecting 30 runs

  prompt 9: collecting 30 runs

  prompt 10: collecting 30 runs

  prompt 11: collecting 30 runs

  prompt 12: collecting 30 runs

  prompt 13: collecting 30 runs

  prompt 14: collecting 30 runs

  prompt 15: collecting 30 runs
Saved prompt 5 run 01 (tokens 1261)
Saved prompt 4 run 01 (tokens 1378)
Saved prompt 1 run 01 (tokens 1590)
Saved prompt 3 run 01 (tokens 1758)
Saved prompt 2 run 01 (tokens 1601)
Saved prompt 6 run 01 (tokens 1573)
Saved prompt 8 run 01 (tokens 1353)
Saved prompt 10 run 01 (tokens 1204)
Saved prompt 9 run 01 (tokens 1356)
Saved prompt 7 run 01 (tokens 1252)
Saved prompt 12 run 01 (tokens 1220)
Saved prompt 11 run 01 (tokens 1447)
Saved prompt 14 run 01 (tokens 1404)
Saved prompt 13 run 01 (tokens 1453)
Saved

## 4. Reproducibility — one task per variant, 30 sequential runs with 60s spacing

In [25]:
async def run_variant(prompt_id, variant_id, variant_text, sem, csv_lock):
    print(f'\n  prompt {prompt_id}:{variant_id} collecting {NUM_RUNS} runs')
    for run_id in range(1, NUM_RUNS + 1):
        retries = 0
        while True:
            try:
                out = await call_model(variant_text, sem)
                row = [
                    prompt_id, variant_id, variant_text, run_id,
                    MODEL_NAME, TEMPERATURE,
                    datetime.datetime.utcnow().isoformat(),
                    out['latency_ms'],
                    out['prompt_tokens'], out['completion_tokens'], out['total_tokens'],
                    out['finish_reason'], out['answer'],
                ]
                async with csv_lock:
                    with open(REPRO_CSV, 'a', newline='', encoding='utf-8') as f:
                        csv.writer(f, quoting=csv.QUOTE_ALL).writerow(row)
                print(f'Saved {prompt_id}:{variant_id} run {run_id:02d} (tokens {out["total_tokens"]})')
                break
            except Exception as e:
                retries += 1
                print(f'Error {prompt_id}:{variant_id} run {run_id:02d} attempt {retries}: {e}')
                if retries >= MAX_RETRIES:
                    print(f'Giving up on {prompt_id}:{variant_id} run {run_id:02d} after {MAX_RETRIES} attempts')
                    break
                await asyncio.sleep(RETRY_SLEEP_SECONDS)
        await asyncio.sleep(INTER_RUN_SLEEP_SECONDS)

async def run_entire_reproducibility():
    sem = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    csv_lock = asyncio.Lock()
    tasks = []
    for prompt_id, variants in PROMPTS_VARIANTS.items():
        for variant_id, variant_text in variants:
            tasks.append(asyncio.create_task(run_variant(int(prompt_id), variant_id, variant_text, sem, csv_lock)))
    await asyncio.gather(*tasks)
    print(f'\n✅ Reproducibility done. Results in {REPRO_CSV}')

await run_entire_reproducibility()


  prompt 1:v1 collecting 30 runs

  prompt 1:v2 collecting 30 runs

  prompt 1:v3 collecting 30 runs

  prompt 1:v4 collecting 30 runs

  prompt 1:v5 collecting 30 runs

  prompt 2:v1 collecting 30 runs

  prompt 2:v2 collecting 30 runs

  prompt 2:v3 collecting 30 runs

  prompt 2:v4 collecting 30 runs

  prompt 2:v5 collecting 30 runs

  prompt 3:v1 collecting 30 runs

  prompt 3:v2 collecting 30 runs

  prompt 3:v3 collecting 30 runs

  prompt 3:v4 collecting 30 runs

  prompt 3:v5 collecting 30 runs

  prompt 4:v1 collecting 30 runs

  prompt 4:v2 collecting 30 runs

  prompt 4:v3 collecting 30 runs

  prompt 4:v4 collecting 30 runs

  prompt 4:v5 collecting 30 runs

  prompt 5:v1 collecting 30 runs

  prompt 5:v2 collecting 30 runs

  prompt 5:v3 collecting 30 runs

  prompt 5:v4 collecting 30 runs

  prompt 5:v5 collecting 30 runs

  prompt 6:v1 collecting 30 runs

  prompt 6:v2 collecting 30 runs

  prompt 6:v3 collecting 30 runs

  prompt 6:v4 collecting 30 runs

  prompt 6:v5

## 6. RESUME CELL — pick up where you left off and stop cleanly on rate limits

This cell:
- Reads whatever's already in both CSVs
- Figures out which (prompt_id, run_id) and (prompt_id, variant_id, run_id) combos are still missing
- Runs ONLY those missing ones
- Detects quota / rate-limit errors (429 or `RESOURCE_EXHAUSTED`) and stops the whole collection cleanly instead of retrying for 6 attempts × 10 seconds each (which would just burn wall-clock time)
- Appends new rows to the same CSVs as before

Safe to re-run multiple times across days — it will skip anything already collected.

In [26]:
# ===== Rate-limit-aware resume logic =====

# Detect quota errors so we stop immediately instead of pointless retries
def is_quota_error(exc):
    msg = str(exc).lower()
    return ('429' in msg or 'resource_exhausted' in msg or 'quota' in msg or 'rate limit' in msg)

class QuotaExceeded(Exception):
    pass

# Build sets of already-done work from the CSVs
def load_done(csv_path, key_cols):
    if not Path(csv_path).exists():
        return set()
    df = pd.read_csv(csv_path)
    return set(df[list(key_cols)].itertuples(index=False, name=None))

done_repeat = load_done(REPEAT_CSV, ('prompt_id', 'run_id'))
done_repro  = load_done(REPRO_CSV,  ('prompt_id', 'variant_id', 'run_id'))

print(f'Repeatability already done: {len(done_repeat)} / 450')
print(f'Reproducibility already done: {len(done_repro)} / 2250')

# ---------- Modified single-call that raises QuotaExceeded on 429 ----------
async def call_model_resume(prompt_text, sem):
    t0 = time.time()
    history = [types.Content(role='user', parts=[types.Part.from_text(text=prompt_text)])]
    collected = []
    tp, tc, tt = 0, 0, 0
    finish_name = None

    async with sem:
        for _ in range(MAX_CONTINUES + 1):
            try:
                resp = await client.aio.models.generate_content(
                    model=MODEL_NAME,
                    contents=history,
                    config=types.GenerateContentConfig(
                        temperature=TEMPERATURE,
                        max_output_tokens=MAX_OUTPUT_TOKENS,
                    ),
                )
            except Exception as e:
                if is_quota_error(e):
                    raise QuotaExceeded(str(e))
                raise  # other errors go to the normal retry logic

            text = _extract_text(resp)
            collected.append(text)

            usage = resp.usage_metadata
            tp += getattr(usage, 'prompt_token_count', 0) or 0
            tc += getattr(usage, 'candidates_token_count', 0) or 0
            tt  = getattr(usage, 'total_token_count', tp + tc) or (tp + tc)

            if resp.candidates:
                fr = resp.candidates[0].finish_reason
                finish_name = fr.name if hasattr(fr, 'name') else str(fr)
            else:
                finish_name = 'UNKNOWN'

            if finish_name != 'MAX_TOKENS':
                break
            history.append(types.Content(role='model', parts=[types.Part.from_text(text=text)]))
            history.append(types.Content(role='user',  parts=[types.Part.from_text(text='Continue.')]))

    t1 = time.time()
    return {
        'answer': '\n'.join([c for c in collected if c]).strip(),
        'finish_reason': finish_name,
        'latency_ms': int((t1 - t0) * 1000),
        'prompt_tokens': tp, 'completion_tokens': tc, 'total_tokens': tt,
    }

# ---------- Shared stop flag so one task can stop the others ----------
stop_flag = asyncio.Event()

async def run_one_resume(write_row_fn, prompt_text, sem, csv_lock, label):
    """Runs one API call with retry. Returns True if successful, False if quota hit."""
    retries = 0
    while True:
        if stop_flag.is_set():
            return False
        try:
            out = await call_model_resume(prompt_text, sem)
            async with csv_lock:
                write_row_fn(out)
            print(f'Saved {label} (tokens {out["total_tokens"]})')
            return True
        except QuotaExceeded as e:
            print(f'🛑 QUOTA EXCEEDED on {label}: {e}')
            print('🛑 Stopping all collection — re-run this cell tomorrow to resume.')
            stop_flag.set()
            return False
        except Exception as e:
            retries += 1
            print(f'Error {label} attempt {retries}: {e}')
            if retries >= MAX_RETRIES:
                print(f'Giving up on {label} after {MAX_RETRIES} attempts')
                return False
            await asyncio.sleep(RETRY_SLEEP_SECONDS)

# ---------- Repeatability resume task ----------
async def resume_prompt_repeatability(prompt_id, prompt_text, sem, csv_lock):
    for run_id in range(1, NUM_RUNS + 1):
        if stop_flag.is_set():
            return
        if (prompt_id, run_id) in done_repeat:
            continue  # already have this row
        def write_row(out, pid=prompt_id, pt=prompt_text, rid=run_id):
            with open(REPEAT_CSV, 'a', newline='', encoding='utf-8') as f:
                csv.writer(f, quoting=csv.QUOTE_ALL).writerow([
                    pid, pt, rid, MODEL_NAME, TEMPERATURE,
                    datetime.datetime.utcnow().isoformat(),
                    out['latency_ms'],
                    out['prompt_tokens'], out['completion_tokens'], out['total_tokens'],
                    out['finish_reason'], out['answer'],
                ])
        ok = await run_one_resume(write_row, prompt_text, sem, csv_lock,
                                   f'prompt {prompt_id} run {run_id:02d}')
        if not ok and stop_flag.is_set():
            return
        await asyncio.sleep(INTER_RUN_SLEEP_SECONDS)

# ---------- Reproducibility resume task ----------
async def resume_variant(prompt_id, variant_id, variant_text, sem, csv_lock):
    for run_id in range(1, NUM_RUNS + 1):
        if stop_flag.is_set():
            return
        if (prompt_id, variant_id, run_id) in done_repro:
            continue
        def write_row(out, pid=prompt_id, vid=variant_id, vt=variant_text, rid=run_id):
            with open(REPRO_CSV, 'a', newline='', encoding='utf-8') as f:
                csv.writer(f, quoting=csv.QUOTE_ALL).writerow([
                    pid, vid, vt, rid, MODEL_NAME, TEMPERATURE,
                    datetime.datetime.utcnow().isoformat(),
                    out['latency_ms'],
                    out['prompt_tokens'], out['completion_tokens'], out['total_tokens'],
                    out['finish_reason'], out['answer'],
                ])
        ok = await run_one_resume(write_row, variant_text, sem, csv_lock,
                                   f'{prompt_id}:{variant_id} run {run_id:02d}')
        if not ok and stop_flag.is_set():
            return
        await asyncio.sleep(INTER_RUN_SLEEP_SECONDS)

# ---------- Run everything that's still missing ----------
async def resume_all():
    sem = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    csv_lock = asyncio.Lock()

    tasks = []
    # Repeatability tasks
    for (pid, ptext) in prompts:
        pid = int(pid)
        if any((pid, rid) not in done_repeat for rid in range(1, NUM_RUNS + 1)):
            tasks.append(asyncio.create_task(resume_prompt_repeatability(pid, ptext, sem, csv_lock)))
    # Reproducibility tasks
    for prompt_id, variants in PROMPTS_VARIANTS.items():
        pid = int(prompt_id)
        for vid, vtext in variants:
            if any((pid, vid, rid) not in done_repro for rid in range(1, NUM_RUNS + 1)):
                tasks.append(asyncio.create_task(resume_variant(pid, vid, vtext, sem, csv_lock)))

    print(f'Scheduled {len(tasks)} resume tasks (prompts × variants that still have missing runs)')
    await asyncio.gather(*tasks)

    if stop_flag.is_set():
        print('\n⏸  Collection paused due to quota. Re-run this cell when quota resets (typically midnight Pacific time).')
    else:
        # Final counts
        rc = pd.read_csv(REPEAT_CSV)
        rp = pd.read_csv(REPRO_CSV)
        print(f'\n✅ All collection complete.')
        print(f'   Repeatability: {len(rc)} / 450 rows')
        print(f'   Reproducibility: {len(rp)} / 2250 rows')

await resume_all()

Repeatability already done: 450 / 450
Reproducibility already done: 2250 / 2250
Scheduled 0 resume tasks (prompts × variants that still have missing runs)

✅ All collection complete.
   Repeatability: 450 / 450 rows
   Reproducibility: 2250 / 2250 rows


## 5. Sanity checks

In [27]:
repeat_check = pd.read_csv(REPEAT_CSV)
repro_check  = pd.read_csv(REPRO_CSV)
print(f'Repeatability: {len(repeat_check)} rows (expected 450)')
print(f'Reproducibility: {len(repro_check)} rows (expected 2250)')
print(f'\nRepeat — finish reasons: {repeat_check["finish_reason"].value_counts().to_dict()}')
print(f'Repro  — finish reasons: {repro_check["finish_reason"].value_counts().to_dict()}')
print(f'\nMean answer length (repeat): {repeat_check["answer_content"].str.len().mean():.0f} chars')
print(f'Mean answer length (repro):  {repro_check["answer_content"].str.len().mean():.0f} chars')

Repeatability: 450 rows (expected 450)
Reproducibility: 2250 rows (expected 2250)

Repeat — finish reasons: {'STOP': 450}
Repro  — finish reasons: {'STOP': 2250}

Mean answer length (repeat): 3700 chars
Mean answer length (repro):  4624 chars
